In [1]:
# Imports
import numpy as np
from surprise import Dataset, Reader, SVD
from surprise.model_selection import cross_validate
import pandas as pd
import pickle

# Load ratings
ratings = pd.read_csv('../data/processed/merged.csv')
ratings = ratings[['userId', 'movieId', 'rating']]
print(ratings.shape)

(1000209, 3)


In [2]:
# Surprise format
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings, reader)

# Train SVD
trainset = data.build_full_trainset()
svd = SVD(n_factors=100, n_epochs=20, random_state=42)
svd.fit(trainset)
print("SVD trained.")

SVD trained.


In [3]:
# Quick cross-validation
results = cross_validate(svd, data, measures=['RMSE'], cv=3, verbose=True)

Evaluating RMSE of algorithm SVD on 3 split(s).

                  Fold 1  Fold 2  Fold 3  Mean    Std     
RMSE (testset)    0.8862  0.8859  0.8861  0.8861  0.0001  
Fit time          4.68    4.52    4.43    4.54    0.11    
Test time         2.08    1.62    1.85    1.85    0.19    


In [4]:
# Top N recommendations for a user
def get_svd_recommendations(user_id, n=10):
    all_movie_ids = ratings['movieId'].unique()
    rated = ratings[ratings['userId'] == user_id]['movieId'].tolist()
    unrated = [m for m in all_movie_ids if m not in rated]
    
    predictions = [(m, svd.predict(user_id, m).est) for m in unrated]
    predictions.sort(key=lambda x: x[1], reverse=True)
    top_ids = [p[0] for p in predictions[:n]]
    
    movies = pd.read_csv('../data/processed/movies.csv')
    return movies[movies['movieId'].isin(top_ids)][['title', 'genres']]

In [5]:
# Test
print(get_svd_recommendations(user_id=1))

# Save
pickle.dump(svd, open('../models/svd_model.pkl', 'wb'))
print("Saved.")

                                                  title            genres
49                           Usual Suspects, The (1995)    Crime|Thriller
315                    Shawshank Redemption, The (1994)             Drama
892                                  Rear Window (1954)  Mystery|Thriller
1186                          Lawrence of Arabia (1962)     Adventure|War
1252                                      Patton (1970)         Drama|War
1499           Shall We Dance? (Shall We Dansu?) (1996)            Comedy
1950  Seven Samurai (The Magnificent Seven) (Shichin...      Action|Drama
2836                                     Sanjuro (1962)  Action|Adventure
3078                             Green Mile, The (1999)    Drama|Thriller
3732                         Anatomy of a Murder (1959)     Drama|Mystery
Saved.
